# 01. TGN Walk-Forward Validation

4년 학습 / 1년 검증 / 1년 테스트 rolling window로 TGN을 평가합니다. `others` 모드(요인 노드 25개 예측)와 `core` 모드(투자자산 7개 예측)를 각각 실행합니다.


In [ ]:
!pip install torch_geometric
!pip install pyinform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 3.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastAggregator,
    TGNMemory,
)
from torch_geometric.nn.conv import TransformerConv
from pyinform.transferentropy import transfer_entropy
import random

## OTHERS 모드


In [ ]:


# ==========================================
# 0. 시드(Seed) 고정 (재현성 확보)
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. 환경 설정 및 하이퍼파라미터
# ==========================================
memory_dim = 100
time_dim = 100
msg_dim = 2
time_window = 30
rolling_window = 60
threshold = 0.001
epochs = 50
target_col_idx = 0
num_classes = 12

investable_assets = [
    'gold', 'vnq', 'xlk', 'xlf', 'xlv', 'xle', 'xly'
]

PREDICT_MODE = 'others'

train_start = '2010-04-01'
test_end = '2018-12-31'

roll_train_years = 4
roll_val_years = 1
roll_test_years = 1
roll_step_years = 1

# ==========================================
# 2. TGN 모델 정의
# ==========================================
class GraphEmbedding(torch.nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_dim):
        super().__init__()
        self.time_enc = Linear(1, msg_dim)
        self.conv = TransformerConv(
            in_channels=in_channels, out_channels=out_channels // 2,
            heads=2, dropout=0.1, edge_dim=msg_dim
        )
    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = (t - last_update[edge_index[0]]).float()
        rel_t_emb = self.time_enc(rel_t.view(-1, 1))
        edge_attr = msg + rel_t_emb
        return self.conv(x, edge_index, edge_attr)

class NodePredictor(torch.nn.Module):
    def __init__(self, embedding_dim, out_dim=12):
        super().__init__()
        self.lin1 = Linear(embedding_dim, embedding_dim // 2)
        self.lin2 = Linear(embedding_dim // 2, out_dim)

    def forward(self, target_node):
        h = self.lin1(target_node)
        h = torch.relu(h)
        return self.lin2(h)

# ==========================================
# 3. 데이터 로더 및 전처리
# ==========================================
def load_csv_data(data_folder='./'):
    csv_files = glob.glob(os.path.join(data_folder, '**', '*.csv'), recursive=True)
    raw_data_dict = {}
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        asset_name = os.path.splitext(file_name)[0]
        if asset_name.lower() == 'manifest': continue
        try:
            df = pd.read_csv(file_path)
            df.columns = df.columns.str.strip()
            rename_dict = {col: 'Date' for col in df.columns if col.lower() in ['date', '날짜', 'time']}
            df.rename(columns=rename_dict, inplace=True)
            raw_data_dict[asset_name] = df
        except Exception as e:
            print(f"[{asset_name}] 파일 로드 에러: {e}")
    return raw_data_dict

def preprocess_data(raw_data_dict):
    if 'vkospi' in raw_data_dict: del raw_data_dict['vkospi']

    asset_names = list(raw_data_dict.keys())
    processed_data = {}

    for name in asset_names:
        df = raw_data_dict[name].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').set_index('Date')

        val_cols = [c for c in df.columns]
        target_col = val_cols[target_col_idx]

        if df[target_col].min() >= 0 and df[target_col].mean() > 1.0:
            df['Log_Return'] = np.log(df[target_col] / df[target_col].shift(1))
        else:
            df['Log_Return'] = df[target_col]

        df['rolling_mean'] = df['Log_Return'].rolling(window=rolling_window).mean()
        df['rolling_std'] = df['Log_Return'].rolling(window=rolling_window).std()
        df['scaled_value'] = (df['Log_Return'] - df['rolling_mean']) / (df['rolling_std'] + 1e-8)

        for col in val_cols:
            if col != target_col:
                c_mean = df[col].rolling(window=rolling_window).mean()
                c_std = df[col].rolling(window=rolling_window).std()
                df[f'scaled_{col}'] = (df[col] - c_mean) / (c_std + 1e-8)

        df = df.dropna()

        raw_class = np.floor(df['scaled_value'])
        clipped_class = raw_class.clip(-6, 5)
        df['target_class'] = (clipped_class + 6).astype(int)

        df = df[(df.index >= train_start) & (df.index <= test_end)]
        processed_data[name] = df

    all_dates = set()
    for df in processed_data.values():
        all_dates.update(df.index)

    global_calendar = pd.Series(list(all_dates)).sort_values().reset_index(drop=True)
    return global_calendar, processed_data, asset_names

# ==========================================
# 4. 동적 네트워크 사전 계산 (🚀 억지 보정 완전 제거!)
# ==========================================
def precompute_dynamic_networks(global_calendar, processed_data, asset_names, time_window, threshold):
    print(f"⏳ 동적 네트워크 사전 계산 시작 (총 {len(global_calendar) - time_window} 스텝, 전체 기간 대상)...")
    start_time = time.time()
    network_dict = {}

    bins = [-1.0, 1.0]

    for t_idx in range(time_window, len(global_calendar) - 1):
        current_date = global_calendar[t_idx]

        active_nodes = []
        for i, name in enumerate(asset_names):
            if current_date in processed_data[name].index and not pd.isna(processed_data[name].loc[current_date, 'scaled_value']):
                active_nodes.append(i)

        sources, targets, edge_features = [], [], []
        if len(active_nodes) > 0:

            for i in active_nodes:
                name_i = asset_names[i]
                current_return = processed_data[name_i].loc[current_date, 'scaled_value']

                # Self-edge (자신으로의 연결)
                sources.append(i); targets.append(i); edge_features.append([current_return, 1.0])

                data_i = processed_data[name_i].loc[:current_date, 'scaled_value'].tail(time_window).values

                for j in active_nodes:
                    if i == j: continue
                    name_j = asset_names[j]

                    data_j = processed_data[name_j].loc[:current_date, 'scaled_value'].tail(time_window).values

                    if len(data_i) == time_window and len(data_j) == time_window:
                        source_discrete = np.digitize(data_i, bins)
                        target_discrete = np.digitize(data_j, bins)

                        te_value = transfer_entropy(source_discrete, target_discrete, k=1)
                        if te_value >= threshold:
                            sources.append(i); targets.append(j); edge_features.append([current_return, te_value])

        if len(sources) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, msg_dim), dtype=torch.float)
        else:
            edge_index = torch.tensor([sources, targets], dtype=torch.long)
            edge_attr = torch.tensor(edge_features, dtype=torch.float)

        network_dict[t_idx] = (active_nodes, edge_index, edge_attr)
        if (t_idx % 500 == 0): print(f"  - 진행률: {t_idx}/{len(global_calendar)} 완료...")

    print(f"✅ 사전 계산 완료 (소요 시간: {time.time() - start_time:.2f}초)")
    return network_dict

# ==========================================
# 5. 메인 학습 (Sliding Window Walk-Forward)
# ==========================================
def main():
    print(f"🚀 장치 설정: {device}")

    my_data_dir = '/content/drive/MyDrive/Find-A final'
    raw_data_dict = load_csv_data(my_data_dir)
    if len(raw_data_dict) == 0: return

    global_calendar, processed_data, asset_names = preprocess_data(raw_data_dict)
    num_asset = len(asset_names)
    all_nodes_tensor = torch.arange(num_asset, dtype=torch.long).to(device)
    criterion = torch.nn.CrossEntropyLoss()

    print(f"📊 로드된 전체 자산 수: {num_asset}개")
    print(f"🎯 현재 예측 모드: '{PREDICT_MODE}'")
    print(f"🗓️ 전체 가용 데이터 기간: {global_calendar.min().date()} ~ {global_calendar.max().date()}\n")

    network_cache = precompute_dynamic_networks(global_calendar, processed_data, asset_names, time_window, threshold)

    overall_start_date = global_calendar.min()
    overall_max_date = global_calendar.max()
    step = 0
    final_test_results = []

    while True:
        t_start = overall_start_date + pd.DateOffset(years=(step * roll_step_years))
        t_end   = t_start + pd.DateOffset(years=roll_train_years) - pd.Timedelta(days=1)
        v_end   = t_end + pd.DateOffset(years=roll_val_years)
        te_end  = v_end + pd.DateOffset(years=roll_test_years)

        if te_end > overall_max_date:
            te_end = overall_max_date

        if v_end >= overall_max_date:
            break

        idx_t_start = global_calendar[global_calendar >= t_start].index[0]
        idx_t_start = max(time_window, idx_t_start)

        idx_t_end  = global_calendar[global_calendar <= t_end].index[-1]
        idx_v_end  = global_calendar[global_calendar <= v_end].index[-1]
        idx_te_end = global_calendar[global_calendar <= te_end].index[-1]

        print("="*60)
        print(f"🔄 [Rolling Window Step {step+1}]")
        print(f"  - Train : {global_calendar.iloc[idx_t_start].date()} ~ {global_calendar.iloc[idx_t_end].date()}")
        print(f"  - Val   : {global_calendar.iloc[idx_t_end+1].date()} ~ {global_calendar.iloc[idx_v_end].date()}")
        print(f"  - Test  : {global_calendar.iloc[idx_v_end+1].date()} ~ {global_calendar.iloc[idx_te_end].date()}")
        print("="*60)

        seed_everything(42 + step)
        memory = TGNMemory(num_nodes=num_asset, raw_msg_dim=msg_dim, memory_dim=memory_dim, time_dim=time_dim,
                           message_module=IdentityMessage(msg_dim, memory_dim, time_dim), aggregator_module=LastAggregator()).to(device)
        gnn = GraphEmbedding(memory_dim, memory_dim, msg_dim, time_dim).to(device)
        predictor = NodePredictor(embedding_dim=memory_dim, out_dim=num_classes).to(device)
        optimizer = torch.optim.AdamW(set(memory.parameters()) | set(gnn.parameters()) | set(predictor.parameters()), lr=0.0001, weight_decay=1e-4)

        best_val_acc = 0.0
        best_memory_state, best_gnn_state, best_predictor_state = None, None, None

        for epoch in range(epochs):
            # --- 1. TRAIN ---
            memory.train(); gnn.train(); predictor.train()
            memory.reset_state()
            train_loss, train_steps = 0, 0

            for t_idx in range(idx_t_start, idx_t_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if len(active_nodes) == 0 or edge_index.size(1) == 0: continue

                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                next_date = global_calendar[t_idx + 1]

                targets, valid_nodes = [], []
                for node in active_nodes:
                    name = asset_names[node]

                    is_core = name.lower() in [a.lower() for a in investable_assets]
                    if (PREDICT_MODE == 'core' and is_core) or (PREDICT_MODE == 'others' and not is_core):
                        if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                            targets.append(processed_data[name].loc[next_date, 'target_class'])
                            valid_nodes.append(node)

                if len(valid_nodes) == 0: continue

                y_target = torch.tensor(targets, dtype=torch.long).to(device)
                optimizer.zero_grad()

                z, last_update = memory(all_nodes_tensor)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                y_pred_logits = predictor(h[valid_nodes])

                loss = criterion(y_pred_logits, y_target)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                train_steps += 1
                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)
                memory.detach()

            # --- 2. VALIDATION ---
            memory.eval(); gnn.eval(); predictor.eval()
            val_loss, val_steps = 0, 0
            val_exact_correct, val_dir_correct, val_total = 0, 0, 0

            with torch.no_grad():
                for t_idx in range(idx_t_end, idx_v_end):
                    active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                    if len(active_nodes) == 0 or edge_index.size(1) == 0: continue

                    edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                    next_date = global_calendar[t_idx + 1]

                    targets, valid_nodes = [], []
                    for node in active_nodes:
                        name = asset_names[node]

                        is_core = name.lower() in [a.lower() for a in investable_assets]
                        if (PREDICT_MODE == 'core' and is_core) or (PREDICT_MODE == 'others' and not is_core):
                            if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                                targets.append(processed_data[name].loc[next_date, 'target_class'])
                                valid_nodes.append(node)

                    if len(valid_nodes) == 0: continue

                    y_target = torch.tensor(targets, dtype=torch.long).to(device)
                    z, last_update = memory(all_nodes_tensor)
                    t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                    h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                    y_pred_logits = predictor(h[valid_nodes])

                    preds = y_pred_logits.argmax(dim=1)
                    val_exact_correct += (preds == y_target).sum().item()
                    val_dir_correct += ((preds >= 6) == (y_target >= 6)).sum().item()
                    val_total += y_target.size(0)
                    memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

            exact_acc = (val_exact_correct / val_total) * 100 if val_total > 0 else 0
            dir_acc = (val_dir_correct / val_total) * 100 if val_total > 0 else 0
            avg_train_loss = train_loss / train_steps if train_steps > 0 else 0

            best_mark = "🌟" if exact_acc > best_val_acc else ""
            if exact_acc > best_val_acc:
                best_val_acc = exact_acc
                best_memory_state = copy.deepcopy(memory.state_dict())
                best_gnn_state = copy.deepcopy(gnn.state_dict())
                best_predictor_state = copy.deepcopy(predictor.state_dict())

            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Exact Acc: {exact_acc:.2f}% | Val Dir Acc: {dir_acc:.2f}% {best_mark}")

        # --- 3. TEST (해당 스텝의 실전) ---
        if best_memory_state is not None:
            memory.load_state_dict(best_memory_state)
            gnn.load_state_dict(best_gnn_state)
            predictor.load_state_dict(best_predictor_state)

        memory.eval(); gnn.eval(); predictor.eval()
        memory.reset_state()

        with torch.no_grad():
            # 워밍업
            for t_idx in range(idx_t_start, idx_v_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if len(active_nodes) == 0 or edge_index.size(1) == 0: continue
                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)
                z, _ = memory(all_nodes_tensor)
                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

            # 진짜 테스트 평가
            test_exact, test_dir, test_tot = 0, 0, 0
            asset_exact = {name: 0 for name in asset_names}
            asset_dir = {name: 0 for name in asset_names}
            asset_total_cnt = {name: 0 for name in asset_names}

            for t_idx in range(idx_v_end, idx_te_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if len(active_nodes) == 0 or edge_index.size(1) == 0: continue

                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                next_date = global_calendar[t_idx + 1]

                targets, valid_nodes = [], []
                for node in active_nodes:
                    name = asset_names[node]

                    is_core = name.lower() in [a.lower() for a in investable_assets]
                    if (PREDICT_MODE == 'core' and is_core) or (PREDICT_MODE == 'others' and not is_core):
                        if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                            targets.append(processed_data[name].loc[next_date, 'target_class'])
                            valid_nodes.append(node)

                if len(valid_nodes) == 0: continue

                y_target = torch.tensor(targets, dtype=torch.long).to(device)
                z, last_update = memory(all_nodes_tensor)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                y_pred_logits = predictor(h[valid_nodes])
                preds = y_pred_logits.argmax(dim=1)

                test_exact += (preds == y_target).sum().item()
                pred_is_up = (preds >= 6)
                target_is_up = (y_target >= 6)
                test_dir += (pred_is_up == target_is_up).sum().item()
                test_tot += y_target.size(0)

                for idx, node in enumerate(valid_nodes):
                    name = asset_names[node]
                    asset_total_cnt[name] += 1
                    if preds[idx].item() == y_target[idx].item(): asset_exact[name] += 1
                    if pred_is_up[idx].item() == target_is_up[idx].item(): asset_dir[name] += 1

                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

        f_exact = (test_exact / test_tot) * 100 if test_tot > 0 else 0
        f_dir = (test_dir / test_tot) * 100 if test_tot > 0 else 0
        print(f"\n🔥 Step {step+1} 실전 결과 👉 완벽 적중률: {f_exact:.2f}% | 방향 적중률: {f_dir:.2f}%")

        indiv_results = []
        for name in asset_names:
            if asset_total_cnt[name] > 0:
                e_acc = (asset_exact[name] / asset_total_cnt[name]) * 100
                d_acc = (asset_dir[name] / asset_total_cnt[name]) * 100
                indiv_results.append(f"{name.upper()}(완벽{e_acc:.1f}%/방향{d_acc:.1f}%)")
        print(f"📊 자산별: {' | '.join(indiv_results)}\n")

        final_test_results.append((f_exact, f_dir))
        step += 1 # 윈도우 이동

    # ==========================================
    # 최종 리포트 출력
    # ==========================================
    print("✨" * 30)
    print(f"📈 [TGN / {PREDICT_MODE.upper()} 모드] Walk-Forward Validation 전체 완료 📈")
    for s, (e, d) in enumerate(final_test_results):
        print(f" - Step {s+1} Test: 완벽 {e:.2f}% | 방향 {d:.2f}%")

    if len(final_test_results) > 0:
        avg_e = sum(x[0] for x in final_test_results) / len(final_test_results)
        avg_d = sum(x[1] for x in final_test_results) / len(final_test_results)
        print("=" * 60)
        print(f"🏆 전체 윈도우 평균 👉 완벽 적중률: {avg_e:.2f}% | 방향 적중률: {avg_d:.2f}%")
    print("✨" * 30)

if __name__ == "__main__":
    main()

🚀 장치 설정: cuda
📊 로드된 전체 자산 수: 32개
🎯 현재 예측 모드: 'others'
🗓️ 전체 가용 데이터 기간: 2010-04-01 ~ 2018-12-31

⏳ 동적 네트워크 사전 계산 시작 (총 2258 스텝, 전체 기간 대상)...
  - 진행률: 500/2288 완료...
  - 진행률: 1000/2288 완료...
  - 진행률: 1500/2288 완료...
  - 진행률: 2000/2288 완료...
✅ 사전 계산 완료 (소요 시간: 475.21초)
🔄 [Rolling Window Step 1]
  - Train : 2010-05-13 ~ 2014-03-31
  - Val   : 2014-04-01 ~ 2015-03-31
  - Test  : 2015-04-01 ~ 2016-03-31
Epoch 01/50 | Train Loss: 1.5280 | Val Exact Acc: 34.51% | Val Dir Acc: 48.89% 🌟
Epoch 02/50 | Train Loss: 1.4657 | Val Exact Acc: 34.56% | Val Dir Acc: 48.98% 🌟
Epoch 03/50 | Train Loss: 1.4609 | Val Exact Acc: 35.58% | Val Dir Acc: 50.26% 🌟
Epoch 04/50 | Train Loss: 1.4579 | Val Exact Acc: 36.54% | Val Dir Acc: 51.81% 🌟
Epoch 05/50 | Train Loss: 1.4557 | Val Exact Acc: 37.70% | Val Dir Acc: 53.61% 🌟
Epoch 06/50 | Train Loss: 1.4502 | Val Exact Acc: 38.48% | Val Dir Acc: 55.67% 🌟
Epoch 07/50 | Train Loss: 1.4407 | Val Exact Acc: 39.37% | Val Dir Acc: 57.16% 🌟
Epoch 08/50 | Train Loss: 1.4305

## CORE 모드


In [ ]:


# ==========================================
# 0. 시드(Seed) 고정 (재현성 확보)
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. 환경 설정 및 하이퍼파라미터
# ==========================================
memory_dim = 100
time_dim = 100
msg_dim = 2
time_window = 30
rolling_window = 60
threshold = 0.001
epochs = 50
target_col_idx = 0
num_classes = 12

investable_assets = [
    'gold', 'vnq', 'xlk', 'xlf', 'xlv', 'xle', 'xly'
]

PREDICT_MODE = 'core'

train_start = '2010-04-01'
test_end = '2018-12-31'

roll_train_years = 4
roll_val_years = 1
roll_test_years = 1
roll_step_years = 1

# ==========================================
# 2. TGN 모델 정의
# ==========================================
class GraphEmbedding(torch.nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_dim):
        super().__init__()
        self.time_enc = Linear(1, msg_dim)
        self.conv = TransformerConv(
            in_channels=in_channels, out_channels=out_channels // 2,
            heads=2, dropout=0.1, edge_dim=msg_dim
        )
    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = (t - last_update[edge_index[0]]).float()
        rel_t_emb = self.time_enc(rel_t.view(-1, 1))
        edge_attr = msg + rel_t_emb
        return self.conv(x, edge_index, edge_attr)

class NodePredictor(torch.nn.Module):
    def __init__(self, embedding_dim, out_dim=12):
        super().__init__()
        self.lin1 = Linear(embedding_dim, embedding_dim // 2)
        self.lin2 = Linear(embedding_dim // 2, out_dim)

    def forward(self, target_node):
        h = self.lin1(target_node)
        h = torch.relu(h)
        return self.lin2(h)

# ==========================================
# 3. 데이터 로더 및 전처리
# ==========================================
def load_csv_data(data_folder='./'):
    csv_files = glob.glob(os.path.join(data_folder, '**', '*.csv'), recursive=True)
    raw_data_dict = {}
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        asset_name = os.path.splitext(file_name)[0]
        if asset_name.lower() == 'manifest': continue
        try:
            df = pd.read_csv(file_path)
            df.columns = df.columns.str.strip()
            rename_dict = {col: 'Date' for col in df.columns if col.lower() in ['date', '날짜', 'time']}
            df.rename(columns=rename_dict, inplace=True)
            raw_data_dict[asset_name] = df
        except Exception as e:
            print(f"[{asset_name}] 파일 로드 에러: {e}")
    return raw_data_dict

def preprocess_data(raw_data_dict):
    if 'vkospi' in raw_data_dict: del raw_data_dict['vkospi']

    asset_names = list(raw_data_dict.keys())
    processed_data = {}

    for name in asset_names:
        df = raw_data_dict[name].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').set_index('Date')

        val_cols = [c for c in df.columns]
        target_col = val_cols[target_col_idx]

        if df[target_col].min() >= 0 and df[target_col].mean() > 1.0:
            df['Log_Return'] = np.log(df[target_col] / df[target_col].shift(1))
        else:
            df['Log_Return'] = df[target_col]

        df['rolling_mean'] = df['Log_Return'].rolling(window=rolling_window).mean()
        df['rolling_std'] = df['Log_Return'].rolling(window=rolling_window).std()
        df['scaled_value'] = (df['Log_Return'] - df['rolling_mean']) / (df['rolling_std'] + 1e-8)

        for col in val_cols:
            if col != target_col:
                c_mean = df[col].rolling(window=rolling_window).mean()
                c_std = df[col].rolling(window=rolling_window).std()
                df[f'scaled_{col}'] = (df[col] - c_mean) / (c_std + 1e-8)

        df = df.dropna()

        raw_class = np.floor(df['scaled_value'])
        clipped_class = raw_class.clip(-6, 5)
        df['target_class'] = (clipped_class + 6).astype(int)

        df = df[(df.index >= train_start) & (df.index <= test_end)]
        processed_data[name] = df

    all_dates = set()
    for df in processed_data.values():
        all_dates.update(df.index)

    global_calendar = pd.Series(list(all_dates)).sort_values().reset_index(drop=True)
    return global_calendar, processed_data, asset_names

# ==========================================
# 4. 동적 네트워크 사전 계산 (🚀 억지 보정 완전 제거!)
# ==========================================
def precompute_dynamic_networks(global_calendar, processed_data, asset_names, time_window, threshold):
    print(f"⏳ 동적 네트워크 사전 계산 시작 (총 {len(global_calendar) - time_window} 스텝, 전체 기간 대상)...")
    start_time = time.time()
    network_dict = {}

    bins = [-1.0, 1.0]

    for t_idx in range(time_window, len(global_calendar) - 1):
        current_date = global_calendar[t_idx]

        active_nodes = []
        for i, name in enumerate(asset_names):
            if current_date in processed_data[name].index and not pd.isna(processed_data[name].loc[current_date, 'scaled_value']):
                active_nodes.append(i)

        sources, targets, edge_features = [], [], []
        if len(active_nodes) > 0:

            for i in active_nodes:
                name_i = asset_names[i]
                current_return = processed_data[name_i].loc[current_date, 'scaled_value']

                # Self-edge (자신으로의 연결)
                sources.append(i); targets.append(i); edge_features.append([current_return, 1.0])

                data_i = processed_data[name_i].loc[:current_date, 'scaled_value'].tail(time_window).values

                for j in active_nodes:
                    if i == j: continue
                    name_j = asset_names[j]

                    data_j = processed_data[name_j].loc[:current_date, 'scaled_value'].tail(time_window).values

                    if len(data_i) == time_window and len(data_j) == time_window:
                        source_discrete = np.digitize(data_i, bins)
                        target_discrete = np.digitize(data_j, bins)

                        te_value = transfer_entropy(source_discrete, target_discrete, k=1)
                        if te_value >= threshold:
                            sources.append(i); targets.append(j); edge_features.append([current_return, te_value])

        if len(sources) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, msg_dim), dtype=torch.float)
        else:
            edge_index = torch.tensor([sources, targets], dtype=torch.long)
            edge_attr = torch.tensor(edge_features, dtype=torch.float)

        network_dict[t_idx] = (active_nodes, edge_index, edge_attr)
        if (t_idx % 500 == 0): print(f"  - 진행률: {t_idx}/{len(global_calendar)} 완료...")

    print(f"✅ 사전 계산 완료 (소요 시간: {time.time() - start_time:.2f}초)")
    return network_dict

# ==========================================
# 5. 메인 학습 (Sliding Window Walk-Forward)
# ==========================================
def main():
    print(f"🚀 장치 설정: {device}")

    my_data_dir = '/content/drive/MyDrive/Find-A final'
    raw_data_dict = load_csv_data(my_data_dir)
    if len(raw_data_dict) == 0: return

    global_calendar, processed_data, asset_names = preprocess_data(raw_data_dict)
    num_asset = len(asset_names)
    all_nodes_tensor = torch.arange(num_asset, dtype=torch.long).to(device)
    criterion = torch.nn.CrossEntropyLoss()

    print(f"📊 로드된 전체 자산 수: {num_asset}개")
    print(f"🎯 현재 예측 모드: '{PREDICT_MODE}'")
    print(f"🗓️ 전체 가용 데이터 기간: {global_calendar.min().date()} ~ {global_calendar.max().date()}\n")

    network_cache = precompute_dynamic_networks(global_calendar, processed_data, asset_names, time_window, threshold)

    overall_start_date = global_calendar.min()
    overall_max_date = global_calendar.max()
    step = 0
    final_test_results = []

    while True:
        t_start = overall_start_date + pd.DateOffset(years=(step * roll_step_years))
        t_end   = t_start + pd.DateOffset(years=roll_train_years) - pd.Timedelta(days=1)
        v_end   = t_end + pd.DateOffset(years=roll_val_years)
        te_end  = v_end + pd.DateOffset(years=roll_test_years)

        if te_end > overall_max_date:
            te_end = overall_max_date

        if v_end >= overall_max_date:
            break

        idx_t_start = global_calendar[global_calendar >= t_start].index[0]
        idx_t_start = max(time_window, idx_t_start)

        idx_t_end  = global_calendar[global_calendar <= t_end].index[-1]
        idx_v_end  = global_calendar[global_calendar <= v_end].index[-1]
        idx_te_end = global_calendar[global_calendar <= te_end].index[-1]

        print("="*60)
        print(f"🔄 [Rolling Window Step {step+1}]")
        print(f"  - Train : {global_calendar.iloc[idx_t_start].date()} ~ {global_calendar.iloc[idx_t_end].date()}")
        print(f"  - Val   : {global_calendar.iloc[idx_t_end+1].date()} ~ {global_calendar.iloc[idx_v_end].date()}")
        print(f"  - Test  : {global_calendar.iloc[idx_v_end+1].date()} ~ {global_calendar.iloc[idx_te_end].date()}")
        print("="*60)

        seed_everything(42 + step)
        memory = TGNMemory(num_nodes=num_asset, raw_msg_dim=msg_dim, memory_dim=memory_dim, time_dim=time_dim,
                           message_module=IdentityMessage(msg_dim, memory_dim, time_dim), aggregator_module=LastAggregator()).to(device)
        gnn = GraphEmbedding(memory_dim, memory_dim, msg_dim, time_dim).to(device)
        predictor = NodePredictor(embedding_dim=memory_dim, out_dim=num_classes).to(device)
        optimizer = torch.optim.AdamW(set(memory.parameters()) | set(gnn.parameters()) | set(predictor.parameters()), lr=0.0001, weight_decay=1e-4)

        best_val_acc = 0.0
        best_memory_state, best_gnn_state, best_predictor_state = None, None, None

        for epoch in range(epochs):
            # --- 1. TRAIN ---
            memory.train(); gnn.train(); predictor.train()
            memory.reset_state()
            train_loss, train_steps = 0, 0

            for t_idx in range(idx_t_start, idx_t_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if len(active_nodes) == 0 or edge_index.size(1) == 0: continue

                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                next_date = global_calendar[t_idx + 1]

                targets, valid_nodes = [], []
                for node in active_nodes:
                    name = asset_names[node]

                    is_core = name.lower() in [a.lower() for a in investable_assets]
                    if (PREDICT_MODE == 'core' and is_core) or (PREDICT_MODE == 'others' and not is_core):
                        if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                            targets.append(processed_data[name].loc[next_date, 'target_class'])
                            valid_nodes.append(node)

                if len(valid_nodes) == 0: continue

                y_target = torch.tensor(targets, dtype=torch.long).to(device)
                optimizer.zero_grad()

                z, last_update = memory(all_nodes_tensor)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                y_pred_logits = predictor(h[valid_nodes])

                loss = criterion(y_pred_logits, y_target)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                train_steps += 1
                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)
                memory.detach()

            # --- 2. VALIDATION ---
            memory.eval(); gnn.eval(); predictor.eval()
            val_loss, val_steps = 0, 0
            val_exact_correct, val_dir_correct, val_total = 0, 0, 0

            with torch.no_grad():
                for t_idx in range(idx_t_end, idx_v_end):
                    active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                    if len(active_nodes) == 0 or edge_index.size(1) == 0: continue

                    edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                    next_date = global_calendar[t_idx + 1]

                    targets, valid_nodes = [], []
                    for node in active_nodes:
                        name = asset_names[node]

                        is_core = name.lower() in [a.lower() for a in investable_assets]
                        if (PREDICT_MODE == 'core' and is_core) or (PREDICT_MODE == 'others' and not is_core):
                            if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                                targets.append(processed_data[name].loc[next_date, 'target_class'])
                                valid_nodes.append(node)

                    if len(valid_nodes) == 0: continue

                    y_target = torch.tensor(targets, dtype=torch.long).to(device)
                    z, last_update = memory(all_nodes_tensor)
                    t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                    h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                    y_pred_logits = predictor(h[valid_nodes])

                    preds = y_pred_logits.argmax(dim=1)
                    val_exact_correct += (preds == y_target).sum().item()
                    val_dir_correct += ((preds >= 6) == (y_target >= 6)).sum().item()
                    val_total += y_target.size(0)
                    memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

            exact_acc = (val_exact_correct / val_total) * 100 if val_total > 0 else 0
            dir_acc = (val_dir_correct / val_total) * 100 if val_total > 0 else 0
            avg_train_loss = train_loss / train_steps if train_steps > 0 else 0

            best_mark = "🌟" if exact_acc > best_val_acc else ""
            if exact_acc > best_val_acc:
                best_val_acc = exact_acc
                best_memory_state = copy.deepcopy(memory.state_dict())
                best_gnn_state = copy.deepcopy(gnn.state_dict())
                best_predictor_state = copy.deepcopy(predictor.state_dict())

            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Exact Acc: {exact_acc:.2f}% | Val Dir Acc: {dir_acc:.2f}% {best_mark}")

        # --- 3. TEST (해당 스텝의 실전) ---
        if best_memory_state is not None:
            memory.load_state_dict(best_memory_state)
            gnn.load_state_dict(best_gnn_state)
            predictor.load_state_dict(best_predictor_state)

        memory.eval(); gnn.eval(); predictor.eval()
        memory.reset_state()

        with torch.no_grad():
            # 워밍업
            for t_idx in range(idx_t_start, idx_v_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if len(active_nodes) == 0 or edge_index.size(1) == 0: continue
                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)
                z, _ = memory(all_nodes_tensor)
                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

            # 진짜 테스트 평가
            test_exact, test_dir, test_tot = 0, 0, 0
            asset_exact = {name: 0 for name in asset_names}
            asset_dir = {name: 0 for name in asset_names}
            asset_total_cnt = {name: 0 for name in asset_names}

            for t_idx in range(idx_v_end, idx_te_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if len(active_nodes) == 0 or edge_index.size(1) == 0: continue

                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                next_date = global_calendar[t_idx + 1]

                targets, valid_nodes = [], []
                for node in active_nodes:
                    name = asset_names[node]

                    is_core = name.lower() in [a.lower() for a in investable_assets]
                    if (PREDICT_MODE == 'core' and is_core) or (PREDICT_MODE == 'others' and not is_core):
                        if next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                            targets.append(processed_data[name].loc[next_date, 'target_class'])
                            valid_nodes.append(node)

                if len(valid_nodes) == 0: continue

                y_target = torch.tensor(targets, dtype=torch.long).to(device)
                z, last_update = memory(all_nodes_tensor)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                y_pred_logits = predictor(h[valid_nodes])
                preds = y_pred_logits.argmax(dim=1)

                test_exact += (preds == y_target).sum().item()
                pred_is_up = (preds >= 6)
                target_is_up = (y_target >= 6)
                test_dir += (pred_is_up == target_is_up).sum().item()
                test_tot += y_target.size(0)

                for idx, node in enumerate(valid_nodes):
                    name = asset_names[node]
                    asset_total_cnt[name] += 1
                    if preds[idx].item() == y_target[idx].item(): asset_exact[name] += 1
                    if pred_is_up[idx].item() == target_is_up[idx].item(): asset_dir[name] += 1

                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

        f_exact = (test_exact / test_tot) * 100 if test_tot > 0 else 0
        f_dir = (test_dir / test_tot) * 100 if test_tot > 0 else 0
        print(f"\n🔥 Step {step+1} 실전 결과 👉 완벽 적중률: {f_exact:.2f}% | 방향 적중률: {f_dir:.2f}%")

        indiv_results = []
        for name in asset_names:
            if asset_total_cnt[name] > 0:
                e_acc = (asset_exact[name] / asset_total_cnt[name]) * 100
                d_acc = (asset_dir[name] / asset_total_cnt[name]) * 100
                indiv_results.append(f"{name.upper()}(완벽{e_acc:.1f}%/방향{d_acc:.1f}%)")
        print(f"📊 자산별: {' | '.join(indiv_results)}\n")

        final_test_results.append((f_exact, f_dir))
        step += 1 # 윈도우 이동

    # ==========================================
    # 최종 리포트 출력
    # ==========================================
    print("✨" * 30)
    print(f"📈 [TGN / {PREDICT_MODE.upper()} 모드] Walk-Forward Validation 전체 완료 📈")
    for s, (e, d) in enumerate(final_test_results):
        print(f" - Step {s+1} Test: 완벽 {e:.2f}% | 방향 {d:.2f}%")

    if len(final_test_results) > 0:
        avg_e = sum(x[0] for x in final_test_results) / len(final_test_results)
        avg_d = sum(x[1] for x in final_test_results) / len(final_test_results)
        print("=" * 60)
        print(f"🏆 전체 윈도우 평균 👉 완벽 적중률: {avg_e:.2f}% | 방향 적중률: {avg_d:.2f}%")
    print("✨" * 30)

if __name__ == "__main__":
    main()

🚀 장치 설정: cuda
📊 로드된 전체 자산 수: 32개
🎯 현재 예측 모드: 'core'
🗓️ 전체 가용 데이터 기간: 2010-04-01 ~ 2018-12-31

⏳ 동적 네트워크 사전 계산 시작 (총 2258 스텝, 전체 기간 대상)...
  - 진행률: 500/2288 완료...
  - 진행률: 1000/2288 완료...
  - 진행률: 1500/2288 완료...
  - 진행률: 2000/2288 완료...
✅ 사전 계산 완료 (소요 시간: 549.90초)
🔄 [Rolling Window Step 1]
  - Train : 2010-05-13 ~ 2014-03-31
  - Val   : 2014-04-01 ~ 2015-03-31
  - Test  : 2015-04-01 ~ 2016-03-31
Epoch 01/50 | Train Loss: 1.5379 | Val Exact Acc: 35.99% | Val Dir Acc: 50.75% 🌟
Epoch 02/50 | Train Loss: 1.4586 | Val Exact Acc: 35.80% | Val Dir Acc: 50.51% 
Epoch 03/50 | Train Loss: 1.4551 | Val Exact Acc: 35.56% | Val Dir Acc: 50.27% 
Epoch 04/50 | Train Loss: 1.4523 | Val Exact Acc: 35.74% | Val Dir Acc: 50.45% 
Epoch 05/50 | Train Loss: 1.4498 | Val Exact Acc: 35.62% | Val Dir Acc: 50.33% 
Epoch 06/50 | Train Loss: 1.4494 | Val Exact Acc: 35.56% | Val Dir Acc: 50.27% 
Epoch 07/50 | Train Loss: 1.4482 | Val Exact Acc: 35.56% | Val Dir Acc: 50.27% 
Epoch 08/50 | Train Loss: 1.4479 | Val E